# Week 3 Day 2: baseline prediction models
Match winner: always predict the home team.

Top player: repeat the previous season's average-fantasy-points leader.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Locate the Day 1 source data whether this notebook is run from Day 2 or the workspace root.
DATA_DIR = Path("../Day 1/afl_datasets")
if not DATA_DIR.exists():
    DATA_DIR = Path("Week 3/Day 1/afl_datasets")

TEAM_FILE = DATA_DIR / "team_matches_home_away_raw - team_matches_home_away_raw.csv.csv"
PLAYER_SEASON_FILE = DATA_DIR / "afl_players_seasonal_stats_raw.csv"

team_match = pd.read_csv(TEAM_FILE, low_memory=False)
player_season = pd.read_csv(PLAYER_SEASON_FILE, low_memory=False)
team_match["match_date"] = pd.to_datetime(team_match["match_date"], errors="coerce")

# Keep only decisive team results for the binary match-winner target.
team_match = team_match[team_match["result"].isin(["W", "L"])].copy()
team_match["team_win_flag"] = (team_match["result"] == "W").astype(int)

# A match appears once from each team's perspective. Use the home perspective as
# the single evaluation row and verify that the reciprocal row agrees.
team_match["match_key"] = (
    team_match["match_date"].dt.strftime("%Y-%m-%d")
    + "|" + team_match["team_name"].astype("string")
    + "|" + team_match["opponent"].astype("string")
    + "|" + team_match["round"].astype("string")
)
match_rows = team_match[team_match["home_away"].astype("string").str.upper().eq("H")].copy()
match_rows = match_rows.drop_duplicates("match_key").sort_values(["match_date", "id"])

# Time-based hold-out avoids training on future matches.
all_seasons = sorted(match_rows["year"].dropna().astype(int).unique())
match_cutoff_index = max(1, int(len(all_seasons) * 0.8))
match_cutoff_year = all_seasons[match_cutoff_index]
match_train = match_rows[match_rows["year"] < match_cutoff_year].copy()
match_test = match_rows[match_rows["year"] >= match_cutoff_year].copy()

# Baseline: home team always wins.
match_test["home_win_prediction"] = 1
match_accuracy = accuracy_score(match_test["team_win_flag"], match_test["home_win_prediction"])
match_f1 = f1_score(match_test["team_win_flag"], match_test["home_win_prediction"], zero_division=0)
try:
    match_roc_auc = roc_auc_score(match_test["team_win_flag"], match_test["home_win_prediction"])
except ValueError:
    match_roc_auc = np.nan

print("MATCH WINNER BASELINE: ALWAYS PREDICT HOME WIN")
print(f"Train seasons: through {match_cutoff_year - 1}; hold-out seasons: {match_cutoff_year}-{all_seasons[-1]}")
print(f"Train matches: {len(match_train):,}; hold-out matches: {len(match_test):,}")
print(f"Accuracy: {match_accuracy:.3f}")
print(f"F1: {match_f1:.3f}")
print(f"ROC AUC: {'N/A for constant predictions' if pd.isna(match_roc_auc) else f'{match_roc_auc:.3f}'}")

# Build one regular-season row per player-season. If finals and regular-season
# rows both exist, aggregate their totals and recompute per-game rates.
player_season = player_season.copy()
player_season["player_id"] = player_season["player_id"].astype("string")
player_season["year"] = pd.to_numeric(player_season["year"], errors="coerce").astype("Int64")
player_season["games_played"] = pd.to_numeric(player_season["games_played"], errors="coerce").fillna(0)
player_season["total_fantasy_points"] = pd.to_numeric(player_season["total_fantasy_points"], errors="coerce")

player_season = player_season[player_season["games_played"] > 0].copy()
season_player = (
    player_season.groupby(["year", "player_id"], as_index=False)
    .agg(
        games_played=("games_played", "sum"),
        total_fantasy_points=("total_fantasy_points", "sum"),
    )
)
season_player["avg_fantasy_points"] = (
    season_player["total_fantasy_points"] / season_player["games_played"]
)

# Define the top-player target as the player with the highest season-average
# fantasy points. Ties are broken deterministically by player ID.
leaders = (
    season_player.sort_values(
        ["year", "avg_fantasy_points", "games_played", "player_id"],
        ascending=[True, False, False, True],
    )
    .drop_duplicates("year")
    .rename(columns={"player_id": "actual_top_player_id"})
)

player_cutoff_year = match_cutoff_year
player_test_years = sorted(
    leaders.loc[leaders["year"] >= player_cutoff_year, "year"].dropna().astype(int).unique()
)

# Previous-season leader baseline. Each hold-out season uses only the most recent
# completed season, including earlier hold-out seasons.
leader_by_year = dict(zip(leaders["year"].astype(int), leaders["actual_top_player_id"]))
leader_years = sorted(leader_by_year)
player_predictions = []
for year in player_test_years:
    previous_years = [candidate for candidate in leader_years if candidate < year]
    if not previous_years:
        continue
    previous_year = max(previous_years)
    actual = leaders.loc[leaders["year"].eq(year)].iloc[0]
    player_predictions.append(
        {
            "year": year,
            "previous_year": previous_year,
            "predicted_top_player_id": leader_by_year[previous_year],
            "actual_top_player_id": actual["actual_top_player_id"],
            "actual_top_score": actual["avg_fantasy_points"],
        }
    )

player_results = pd.DataFrame(player_predictions)
player_results["top_1_correct"] = (
    player_results["predicted_top_player_id"] == player_results["actual_top_player_id"]
)

# Top-k is based on the actual hold-out season leaderboard, not only its winner.
actual_rankings = {
    int(year): group.sort_values(
        ["avg_fantasy_points", "games_played", "player_id"],
        ascending=[False, False, True],
    )["player_id"].tolist()
    for year, group in season_player[season_player["year"].isin(player_test_years)].groupby("year")
}
player_results["top_3_correct"] = player_results.apply(
    lambda row: row["predicted_top_player_id"] in actual_rankings.get(int(row["year"]), [])[:3],
    axis=1,
)

print("\nTOP PLAYER BASELINE: PREVIOUS SEASON'S AVG-FANTASY-POINTS LEADER")
print(f"Hold-out seasons: {player_test_years[0]}-{player_test_years[-1]}")
print(f"Evaluated seasons: {len(player_results)}")
print(f"Top-1 accuracy: {player_results['top_1_correct'].mean():.3f}")
print(f"Top-3 accuracy: {player_results['top_3_correct'].mean():.3f}")
print("\nPer-season results:")
print(player_results.to_string(index=False))

baseline_metrics = pd.DataFrame(
    [
        {"model": "Always home win", "metric": "accuracy", "value": match_accuracy},
        {"model": "Always home win", "metric": "f1", "value": match_f1},
        {"model": "Always home win", "metric": "roc_auc", "value": match_roc_auc},
        {"model": "Previous season top fantasy leader", "metric": "top_1_accuracy", "value": player_results["top_1_correct"].mean()},
        {"model": "Previous season top fantasy leader", "metric": "top_3_accuracy", "value": player_results["top_3_correct"].mean()},
    ]
)
print("\nBASELINE METRICS")
print(baseline_metrics.to_string(index=False))

MATCH WINNER BASELINE: ALWAYS PREDICT HOME WIN
Train seasons: through 2016; hold-out seasons: 2017-2025
Train matches: 6,010; hold-out matches: 1,829
Accuracy: 0.569
F1: 0.725
ROC AUC: 0.500

TOP PLAYER BASELINE: PREVIOUS SEASON'S AVG-FANTASY-POINTS LEADER
Hold-out seasons: 2017-2025
Evaluated seasons: 9
Top-1 accuracy: 0.111
Top-3 accuracy: 0.222

Per-season results:
 year  previous_year predicted_top_player_id actual_top_player_id  actual_top_score  top_1_correct  top_3_correct
 2017           2016                   43674                44510        127.181818          False           True
 2018           2017                   44510                44510        128.166667           True           True
 2019           2018                   44510                43950        123.916667          False          False
 2020           2019                   43950                43760        100.000000          False          False
 2021           2020                   43760               

# TASK 2: MATCH WINNER MODELS


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Derive only pre-match features. The target, score, margin, and quarter-score
# columns are deliberately excluded to prevent same-match leakage.
team_history = team_match.sort_values(["team_name", "match_date", "id"]).copy()
team_history["recent_form_5"] = (
    team_history.groupby("team_name")["team_win_flag"]
    .transform(lambda values: values.shift(1).rolling(5, min_periods=1).mean())
)
team_history["days_rest"] = (
    team_history.groupby("team_name")["match_date"].diff().dt.days
)

home_features = team_history[
    team_history["home_away"].astype("string").str.upper().eq("H")
].copy()
home_features = home_features.rename(
    columns={
        "team_name": "home_team",
        "opponent": "away_team",
        "recent_form_5": "home_recent_form_5",
        "days_rest": "home_days_rest",
    }
)

opponent_lookup = team_history[
    ["match_date", "round", "team_name", "opponent", "recent_form_5", "days_rest"]
].rename(
    columns={
        "team_name": "away_team",
        "opponent": "home_team",
        "recent_form_5": "away_recent_form_5",
        "days_rest": "away_days_rest",
    }
)

model_rows = home_features.merge(
    opponent_lookup,
    on=["match_date", "round", "home_team", "away_team"],
    how="left",
    validate="one_to_one",
)
model_rows["home_recent_form_5"] = model_rows["home_recent_form_5"].fillna(0.5)
model_rows["away_recent_form_5"] = model_rows["away_recent_form_5"].fillna(0.5)
model_rows["home_days_rest"] = model_rows["home_days_rest"].fillna(model_rows["home_days_rest"].median())
model_rows["away_days_rest"] = model_rows["away_days_rest"].fillna(model_rows["away_days_rest"].median())

# Use the same chronological season hold-out as the baseline.
model_train = model_rows[model_rows["year"] < match_cutoff_year].copy()
model_test = model_rows[model_rows["year"] >= match_cutoff_year].copy()

numeric_features = [
    "year",
    "home_recent_form_5",
    "away_recent_form_5",
    "home_days_rest",
    "away_days_rest",
]
categorical_features = ["home_team", "away_team", "venue", "round"]
model_features = numeric_features + categorical_features
X_train = model_train[model_features]
y_train = model_train["team_win_flag"]
X_test = model_test[model_features]
y_test = model_test["team_win_flag"]

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ],
    remainder="drop",
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        random_state=42,
    ),
}

model_pipelines = {
    name: Pipeline([("preprocess", preprocessor), ("model", estimator)])
    for name, estimator in models.items()
}

model_metrics = []
for name, pipeline in model_pipelines.items():
    pipeline.fit(X_train, y_train)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    model_metrics.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, predictions),
            "f1": f1_score(y_test, predictions, zero_division=0),
            "roc_auc": roc_auc_score(y_test, probabilities),
            "brier_score": brier_score_loss(y_test, probabilities),
        }
    )

model_metrics = pd.DataFrame(model_metrics).sort_values(
    ["roc_auc", "brier_score"], ascending=[False, True]
).reset_index(drop=True)

# Select primarily by discrimination, then by probability calibration. This
# keeps the choice reproducible while exposing the calibration trade-off.
final_model_name = model_metrics.iloc[0]["model"]
final_model = model_pipelines[final_model_name]

print("MATCH WINNER MODEL EVALUATION")
print(f"Train seasons: through {match_cutoff_year - 1}; hold-out seasons: {match_cutoff_year}-{all_seasons[-1]}")
print(f"Train rows: {len(model_train):,}; hold-out rows: {len(model_test):,}")
print("\nFeatures:", model_features)
print("\nMetrics (higher is better except Brier score):")
print(model_metrics.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
print(f"\nFINAL MODEL: {final_model_name}")
print("Selection rule: highest hold-out ROC AUC, with lower Brier score as the tie-breaker.")

# Show interpretable drivers for the selected model where the estimator exposes them.
feature_names = final_model.named_steps["preprocess"].get_feature_names_out()
if final_model_name == "Logistic Regression":
    coefficients = final_model.named_steps["model"].coef_[0]
    interpretation = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
    interpretation["absolute_coefficient"] = interpretation["coefficient"].abs()
    interpretation = interpretation.sort_values("absolute_coefficient", ascending=False).head(12)
    print("\nTOP LOGISTIC REGRESSION DRIVERS:")
    print(interpretation[["feature", "coefficient"]].to_string(index=False))
else:
    importances = final_model.named_steps["model"].feature_importances_
    interpretation = pd.DataFrame({"feature": feature_names, "importance": importances})
    interpretation = interpretation.sort_values("importance", ascending=False).head(12)
    print("\nTOP GRADIENT BOOSTING DRIVERS:")
    print(interpretation.to_string(index=False))

print("\nINTERPRETABILITY TRADE-OFF:")
print("Logistic Regression provides signed, easy-to-explain feature effects and usually simpler probability behavior.")
print("Gradient Boosting captures nonlinear interactions and thresholds but is less transparent; inspect feature importance and calibration before deployment.")

MATCH WINNER MODEL EVALUATION
Train seasons: through 2016; hold-out seasons: 2017-2025
Train rows: 6,010; hold-out rows: 1,829

Features: ['year', 'home_recent_form_5', 'away_recent_form_5', 'home_days_rest', 'away_days_rest', 'home_team', 'away_team', 'venue', 'round']

Metrics (higher is better except Brier score):
              model  accuracy    f1  roc_auc  brier_score
  Gradient Boosting     0.614 0.704    0.638        0.231
Logistic Regression     0.577 0.589    0.612        0.250

FINAL MODEL: Gradient Boosting
Selection rule: highest hold-out ROC AUC, with lower Brier score as the tie-breaker.

TOP GRADIENT BOOSTING DRIVERS:
                               feature  importance
           numeric__home_recent_form_5    0.430253
           numeric__away_recent_form_5    0.196783
                         numeric__year    0.062778
                 categorical__round_SF    0.031273
   categorical__home_team_Geelong Cats    0.023255
                 categorical__round_QF    0.022501
 

# TASK 3: TOP PLAYER REGRESSION MODEL


Framing choice: regression predicts a continuous fantasy score for every
player-match row, then ranking those predictions identifies the top performer.
This is preferable here to learning-to-rank because the source has a natural
numeric target and the predicted score can be reused by the future chat agent.

In [4]:
# ============================================================
# TASK 3: TOP PLAYER REGRESSION MODEL
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder

# Framing choice: regression predicts a continuous fantasy score for every
# player-match row, then ranking those predictions identifies the top performer.
# This is preferable here to learning-to-rank because the source has a natural
# numeric target and the predicted score can be reused by the future chat agent.
player_match_file = DATA_DIR / "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv"
player_match_model = pd.read_csv(player_match_file, low_memory=False)
player_match_model["match_date"] = pd.to_datetime(player_match_model["match_date"], errors="coerce")
player_match_model["player_id"] = player_match_model["player_id"].astype("string")
player_match_model["team"] = player_match_model["team"].astype("string")
player_match_model["opponent"] = player_match_model["opponent"].astype("string")
player_match_model["fantasy_points"] = pd.to_numeric(player_match_model["fantasy_points"], errors="coerce")
player_match_model = player_match_model.dropna(subset=["match_date", "fantasy_points", "player_id"])

# Canonical game key lets both team perspectives be evaluated as one contest.
player_match_model["game_key"] = (
    player_match_model["match_date"].dt.strftime("%Y-%m-%d")
    + "|" + player_match_model[["team", "opponent"]].apply(lambda row: "|".join(sorted(row)), axis=1)
    + "|" + player_match_model["round"].astype("string")
)
player_match_model = player_match_model.sort_values(["player_id", "match_date", "id"])

# All player features are shifted before rolling, so the current match score is
# never available to its own prediction.
for source_column, feature_name in [
    ("fantasy_points", "prior_avg_fantasy_5"),
    ("disposals", "prior_avg_disposals_5"),
    ("goals", "prior_avg_goals_5"),
]:
    prior_values = player_match_model.groupby("player_id")[source_column].shift(1)
    player_match_model[feature_name] = prior_values.groupby(player_match_model["player_id"]).transform(
        lambda values: values.rolling(5, min_periods=1).mean()
    )
player_match_model["prior_games"] = player_match_model.groupby("player_id").cumcount()

regression_numeric_features = [
    "year",
    "prior_avg_fantasy_5",
    "prior_avg_disposals_5",
    "prior_avg_goals_5",
    "prior_games",
]
regression_categorical_features = ["team", "opponent", "round"]
regression_features = regression_numeric_features + regression_categorical_features

regression_rows = player_match_model.dropna(subset=["prior_avg_fantasy_5"]).copy()
regression_train = regression_rows[regression_rows["year"] < match_cutoff_year].copy()
regression_test = regression_rows[regression_rows["year"] >= match_cutoff_year].copy()

regression_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), regression_numeric_features),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            regression_categorical_features,
        ),
    ],
    remainder="drop",
)

player_regressor = Pipeline(
    steps=[
        ("preprocess", regression_preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=30,
                max_depth=12,
                min_samples_leaf=15,
                n_jobs=-1,
                random_state=42,
            ),
        ),
    ]
)

player_regressor.fit(regression_train[regression_features], regression_train["fantasy_points"])
regression_test["predicted_fantasy_points"] = player_regressor.predict(
    regression_test[regression_features]
)

regression_mae = mean_absolute_error(
    regression_test["fantasy_points"], regression_test["predicted_fantasy_points"]
)
regression_rmse = mean_squared_error(
    regression_test["fantasy_points"], regression_test["predicted_fantasy_points"]
) ** 0.5

# Rank predictions within each game and compare with the actual top performer.
predicted_top_5 = (
    regression_test.sort_values(["game_key", "predicted_fantasy_points"], ascending=[True, False])
    .groupby("game_key")
    .head(5)
    .groupby("game_key")["player_id"]
    .apply(list)
)
actual_top_player = (
    regression_test.sort_values(
        ["game_key", "fantasy_points", "player_id"],
        ascending=[True, False, True],
    )
    .drop_duplicates("game_key")
    .set_index("game_key")["player_id"]
)
ranking_results = pd.DataFrame(
    {
        "predicted_top_5": predicted_top_5,
        "actual_top_player_id": actual_top_player,
    }
).dropna()
ranking_results["top_5_hit"] = ranking_results.apply(
    lambda row: row["actual_top_player_id"] in row["predicted_top_5"], axis=1
)
model_top_5_hit_rate = ranking_results["top_5_hit"].mean()

# Day 2 Task 1 baseline: repeat the previous season's top average-fantasy player.
# A baseline hit is counted only when that player is actually in the game's top five.
season_leader_by_year = dict(zip(leaders["year"].astype(int), leaders["actual_top_player_id"]))
leader_years = sorted(season_leader_by_year)
regression_test["previous_season_leader"] = regression_test["year"].map(
    lambda current_year: season_leader_by_year.get(
        max((year for year in leader_years if year < current_year), default=np.nan), np.nan
    )
)
baseline_top_5_hit = (
    regression_test.groupby("game_key")
    .apply(
        lambda game: (
            game["previous_season_leader"].iloc[0] in set(
                game.nlargest(5, "fantasy_points")["player_id"]
            )
        ),
        include_groups=False,
    )
)
baseline_top_5_hit_rate = baseline_top_5_hit.mean()

comparison = pd.DataFrame(
    [
        {"approach": "Random forest regression", "MAE": regression_mae, "RMSE": regression_rmse, "top_5_hit_rate": model_top_5_hit_rate},
        {"approach": "Previous-season leader baseline", "MAE": np.nan, "RMSE": np.nan, "top_5_hit_rate": baseline_top_5_hit_rate},
    ]
)

print("TOP PLAYER MODEL: REGRESSION THEN WITHIN-MATCH RANKING")
print("Framing: predict fantasy points from lagged player form and match context, then rank available players per game.")
print(f"Train seasons: through {match_cutoff_year - 1}; hold-out seasons: {match_cutoff_year}-{all_seasons[-1]}")
print(f"Train player-match rows: {len(regression_train):,}; hold-out rows: {len(regression_test):,}")
print(f"Evaluated games: {len(ranking_results):,}")
print(f"MAE: {regression_mae:.3f}")
print(f"RMSE: {regression_rmse:.3f}")
print(f"Model top-5 hit rate: {model_top_5_hit_rate:.3f}")
print(f"Task 1 baseline top-5 hit rate: {baseline_top_5_hit_rate:.3f}")
print(f"Top-5 improvement: {model_top_5_hit_rate - baseline_top_5_hit_rate:+.3f}")
print("\nCOMPARISON")
print(comparison.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
print("\nCONCLUSION:", "The regression model is meaningfully better on top-5 hit rate." if model_top_5_hit_rate > baseline_top_5_hit_rate else "The regression model does not yet beat the Task 1 top-player baseline; improve features or use learning-to-rank next.")

TOP PLAYER MODEL: REGRESSION THEN WITHIN-MATCH RANKING
Framing: predict fantasy points from lagged player form and match context, then rank available players per game.
Train seasons: through 2016; hold-out seasons: 2017-2025
Train player-match rows: 187,715; hold-out rows: 83,265
Evaluated games: 1,845
MAE: 17.758
RMSE: 22.482
Model top-5 hit rate: 0.560
Task 1 baseline top-5 hit rate: 0.056
Top-5 improvement: +0.504

COMPARISON
                       approach    MAE   RMSE  top_5_hit_rate
       Random forest regression 17.758 22.482           0.560
Previous-season leader baseline    NaN    NaN           0.056

CONCLUSION: The regression model is meaningfully better on top-5 hit rate.


# TASK 4: FEATURE IMPORTANCE AND SANITY CHECKS



Both models use the same transformed feature space, so their drivers can be
compared directly. Coefficients are signed; tree importances are nonnegative.

In [ ]:
logistic_pipeline = model_pipelines["Logistic Regression"]
gb_pipeline = model_pipelines["Gradient Boosting"]
transformed_feature_names = logistic_pipeline.named_steps["preprocess"].get_feature_names_out()

logistic_coefficients = pd.DataFrame(
    {
        "feature": transformed_feature_names,
        "coefficient": logistic_pipeline.named_steps["model"].coef_[0],
    }
)
logistic_coefficients["absolute_effect"] = logistic_coefficients["coefficient"].abs()
logistic_coefficients = logistic_coefficients.sort_values(
    "absolute_effect", ascending=False
).reset_index(drop=True)

gb_importances = pd.DataFrame(
    {
        "feature": transformed_feature_names,
        "importance": gb_pipeline.named_steps["model"].feature_importances_,
    }
).sort_values("importance", ascending=False).reset_index(drop=True)

print("FEATURE IMPORTANCE AND SANITY CHECKS")
print("\nTOP LOGISTIC REGRESSION COEFFICIENTS")
print(logistic_coefficients.head(15)[["feature", "coefficient"]].to_string(index=False))
print("\nTOP GRADIENT BOOSTING IMPORTANCES")
print(gb_importances.head(15).to_string(index=False))

# Football-sense review: recent form and rest are valid pre-match signals;
# team/venue/round are context but can encode era or competition structure.
def feature_sense(feature_name):
    if "recent_form" in feature_name:
        return "sensible: recent pre-match form"
    if "days_rest" in feature_name:
        return "sensible: recovery/rest context"
    if "venue" in feature_name:
        return "sensible but may encode home-ground effects"
    if "home_team" in feature_name or "away_team" in feature_name:
        return "sensible team-strength proxy; not matchup history"
    if "year" in feature_name:
        return "flag: calendar-year/era proxy"
    if "round" in feature_name:
        return "flag: competition-stage proxy"
    return "review"

importance_review = pd.DataFrame(
    {
        "feature": sorted(
            set(logistic_coefficients.head(15)["feature"])
            | set(gb_importances.head(15)["feature"])
        ),
    }
)
importance_review["football_sense"] = importance_review["feature"].map(feature_sense)
print("\nSANITY REVIEW OF TOP FEATURES")
print(importance_review.to_string(index=False))

print("\nLEAKAGE AUDIT")
print("Excluded same-match fields: result, team_score, opponent_score, margin, quarter scores, and post-match player statistics.")
print("No explicit matchup-history feature is present; team IDs may learn persistent team strength, but they are not head-to-head history.")
print("Potential proxies to monitor: year can capture era/rule changes, round can encode finals structure, and venue can partly encode home advantage.")

# Sniff test: select three held-out matches with a range of form differences.
# The manual expectation uses only pre-match information: stronger recent form
# wins the rule when the gap is clear; otherwise the home side gets the edge.
sniff_candidates = model_test.copy()
sniff_candidates["form_gap"] = (
    sniff_candidates["home_recent_form_5"] - sniff_candidates["away_recent_form_5"]
)
sniff_matches = pd.concat(
    [
        sniff_candidates.loc[sniff_candidates["form_gap"].abs().nlargest(1).index],
        sniff_candidates.loc[sniff_candidates["form_gap"].abs().nsmallest(1).index],
        sniff_candidates.loc[(sniff_candidates["form_gap"] - 0.1).abs().nsmallest(1).index],
    ]
).drop_duplicates("match_key").head(3).copy()

sniff_matches["manual_expectation"] = np.where(
    sniff_matches["form_gap"] > 0.10,
    1,
    np.where(sniff_matches["form_gap"] < -0.10, 0, 1),
)
sniff_matches["manual_reason"] = np.where(
    sniff_matches["form_gap"] > 0.10,
    "Home form is clearly stronger",
    np.where(
        sniff_matches["form_gap"] < -0.10,
        "Away form is clearly stronger despite home ground",
        "Form is close, so home-ground advantage is the tie-break",
    ),
)

sniff_features = sniff_matches[model_features]
sniff_matches["logistic_probability_home"] = logistic_pipeline.predict_proba(sniff_features)[:, 1]
sniff_matches["gradient_boosting_probability_home"] = gb_pipeline.predict_proba(sniff_features)[:, 1]
sniff_matches["logistic_prediction"] = (sniff_matches["logistic_probability_home"] >= 0.5).astype(int)
sniff_matches["gradient_boosting_prediction"] = (
    sniff_matches["gradient_boosting_probability_home"] >= 0.5
).astype(int)
sniff_matches["actual_home_win"] = sniff_matches["team_win_flag"]
sniff_matches["model_disagrees_with_manual"] = (
    sniff_matches["gradient_boosting_prediction"] != sniff_matches["manual_expectation"]
)
sniff_matches["manual_disagrees_with_actual"] = (
    sniff_matches["manual_expectation"] != sniff_matches["actual_home_win"]
)

sniff_columns = [
    "match_date",
    "round",
    "home_team",
    "away_team",
    "home_recent_form_5",
    "away_recent_form_5",
    "home_days_rest",
    "away_days_rest",
    "manual_expectation",
    "manual_reason",
    "logistic_probability_home",
    "gradient_boosting_probability_home",
    "gradient_boosting_prediction",
    "actual_home_win",
    "model_disagrees_with_manual",
    "manual_disagrees_with_actual",
]
print("\nSNIFF TEST: THREE HELD-OUT MATCHES")
print(sniff_matches[sniff_columns].to_string(index=False))

# Investigate disagreements rather than hiding them: print the context that can
# explain a disagreement, including form, rest, venue, and competition stage.
disagreements = sniff_matches[
    sniff_matches["model_disagrees_with_manual"]
    | sniff_matches["manual_disagrees_with_actual"]
].copy()
print("\nSNIFF-TEST DISAGREEMENT INVESTIGATION")
if disagreements.empty:
    print("No disagreement in the three selected matches.")
else:
    disagreement_columns = [
        "match_date", "round", "home_team", "away_team", "venue",
        "home_recent_form_5", "away_recent_form_5", "home_days_rest", "away_days_rest",
        "manual_reason", "gradient_boosting_probability_home", "actual_home_win",
    ]
    print(disagreements[disagreement_columns].to_string(index=False))
    print("Interpretation: disagreements are investigated using only pre-match context; actual outcomes are reported for audit, not used as features.")

FEATURE IMPORTANCE AND SANITY CHECKS

TOP LOGISTIC REGRESSION COEFFICIENTS
                                   feature  coefficient
                     categorical__round_SF     1.809101
                     categorical__round_QF     1.401908
                     categorical__round_PF     1.357195
    categorical__away_team_Gold Coast Suns     1.178333
         categorical__venue_Princes Park\n     1.043562
     categorical__away_team_hawthorn hawks    -1.005372
categorical__venue_Sydney Cricket Ground\n    -0.989858
categorical__home_team_\tWest Coast Eagles     0.922640
      categorical__home_team_Fitzroy Lions    -0.913309
  categorical__home_team_West Coast Eagles     0.852007
    categorical__away_team_gold coast suns     0.820647
     categorical__away_team_Brisbane Bears     0.819299
       categorical__venue_TIO Traeger Park    -0.815215
        categorical__venue_Waverley Park\n    -0.782574
           categorical__venue_Princes Park     0.777362

TOP GRADIENT BOOSTING IMPORT

In [6]:
# Follow-up investigation for suspicious sniff-test inputs.
print("DATA-QUALITY FOLLOW-UP")
long_rest_rows = model_test[
    (model_test["home_days_rest"] > 60) | (model_test["away_days_rest"] > 60)
]
print(f"Held-out rows with rest gap > 60 days: {len(long_rest_rows):,}")
print("These are usually season-break gaps, not same-week recovery effects; consider capping or adding an off-season indicator in a later model revision.")

for column in ["team_name", "opponent", "venue"]:
    values = team_match[column].astype("string")
    whitespace_variants = values[values.str.contains(r"^\\s|\\s$|\\t|\\n", regex=True, na=False)]
    print(f"{column}: rows with leading/trailing/tab/newline whitespace = {len(whitespace_variants):,}")

print("The 2025 Brisbane-Melbourne disagreement is therefore plausible rather than direct leakage: the model favored home form and venue, while the 365-day rest value is a feature-quality warning.")

DATA-QUALITY FOLLOW-UP
Held-out rows with rest gap > 60 days: 148
These are usually season-break gaps, not same-week recovery effects; consider capping or adding an off-season indicator in a later model revision.
team_name: rows with leading/trailing/tab/newline whitespace = 0
opponent: rows with leading/trailing/tab/newline whitespace = 0
venue: rows with leading/trailing/tab/newline whitespace = 0
The 2025 Brisbane-Melbourne disagreement is therefore plausible rather than direct leakage: the model favored home form and venue, while the 365-day rest value is a feature-quality warning.


# TASK 5: PACKAGE MODELS AS CALLABLE FUNCTIONS


In [ ]:
import joblib

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
MATCH_PIPELINE_PATH = MODEL_DIR / "match_winner_pipeline.joblib"
PLAYER_PIPELINE_PATH = MODEL_DIR / "top_player_pipeline.joblib"


def _canonical_name(value, values, label):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{label} must be a non-empty name.")
    cleaned = value.strip().casefold()
    matches = [item for item in values if str(item).strip().casefold() == cleaned]
    if not matches:
        raise ValueError(f"Unknown {label} '{value}'. Check the training-data names.")
    return matches[0]


def _validated_date(value, minimum, maximum):
    parsed = pd.to_datetime(value, errors="coerce")
    if pd.isna(parsed):
        raise ValueError("date must be a valid date such as '2025-05-18'.")
    parsed = pd.Timestamp(parsed)
    if parsed < minimum or parsed > maximum:
        raise ValueError(
            f"date must be between {minimum.date()} and {maximum.date()} "
            "because the fitted model has no validated data outside that range."
        )
    return parsed


class MatchWinnerPredictor:
    """Predict team_a (home) versus team_b (away) with a win probability."""

    def __init__(self, pipeline, history, training_rows):
        self.pipeline = pipeline
        self.history = history.copy()
        self.training_rows = training_rows.copy()
        self.teams = list(self.history["team_name"].dropna().unique())
        self.minimum_date = self.history["match_date"].min()
        self.maximum_date = self.history["match_date"].max()
        self.default_round = self.training_rows["round"].mode().iloc[0]

    def _form_and_rest(self, team, target_date):
        prior = self.history[
            (self.history["team_name"] == team)
            & (self.history["match_date"] < target_date)
        ].sort_values(["match_date", "id"])
        form = prior["team_win_flag"].tail(5).mean() if not prior.empty else 0.5
        rest = (target_date - prior["match_date"].iloc[-1]).days if not prior.empty else 7
        return float(form), float(rest)

    def predict_match_winner(self, team_a, team_b, date, venue=None, round_name=None):
        target_date = _validated_date(date, self.minimum_date, self.maximum_date)
        home = _canonical_name(team_a, self.teams, "team_a")
        away = _canonical_name(team_b, self.teams, "team_b")
        if home == away:
            raise ValueError("team_a and team_b must be different teams.")
        home_form, home_rest = self._form_and_rest(home, target_date)
        away_form, away_rest = self._form_and_rest(away, target_date)
        if venue is None:
            venues = self.training_rows.loc[
                self.training_rows["home_team"].astype("string").str.strip()
                == str(home).strip(), "venue"
            ].dropna()
            venue = venues.mode().iloc[0] if not venues.empty else self.training_rows["venue"].mode().iloc[0]
        row = pd.DataFrame([{
            "year": target_date.year,
            "home_recent_form_5": home_form,
            "away_recent_form_5": away_form,
            "home_days_rest": home_rest,
            "away_days_rest": away_rest,
            "home_team": home,
            "away_team": away,
            "venue": venue,
            "round": self.default_round if round_name is None else round_name,
        }])
        probability = float(self.pipeline.predict_proba(row)[0, 1])
        winner = home if probability >= 0.5 else away
        return {
            "home_team": str(home).strip(),
            "away_team": str(away).strip(),
            "winner": str(winner).strip(),
            "home_win_probability": round(probability, 6),
            "away_win_probability": round(1 - probability, 6),
            "date": target_date.strftime("%Y-%m-%d"),
        }


class TopPlayerPredictor:
    """Predict fantasy points and return a ranked player list."""

    def __init__(self, pipeline, player_history):
        self.pipeline = pipeline
        self.history = player_history.copy()
        self.teams = list(self.history["team"].dropna().unique())
        self.match_ids = set(self.history["game_key"].dropna())
        self.minimum_date = self.history["match_date"].min()
        self.maximum_date = self.history["match_date"].max()

    def _features(self, player_id, team, opponent, target_date, round_name):
        prior = self.history[
            (self.history["player_id"] == player_id)
            & (self.history["match_date"] < target_date)
        ].sort_values(["match_date", "id"])
        if prior.empty:
            raise ValueError(f"No prior history for player '{player_id}'.")
        recent = prior.tail(5)
        return {
            "year": target_date.year,
            "prior_avg_fantasy_5": recent["fantasy_points"].mean(),
            "prior_avg_disposals_5": recent["disposals"].mean(),
            "prior_avg_goals_5": recent["goals"].mean(),
            "prior_games": len(prior),
            "team": team,
            "opponent": opponent,
            "round": round_name,
        }

    def predict_top_player(
        self,
        match_id=None,
        team=None,
        opponent=None,
        date=None,
        stat_type="fantasy_points",
        top_k=5,
        round_name=None,
    ):
        if stat_type not in {"fantasy_points", "fantasy", "score"}:
            raise ValueError("Unsupported stat_type. This pipeline supports 'fantasy_points'.")
        if not isinstance(top_k, int) or top_k < 1:
            raise ValueError("top_k must be a positive integer.")
        if match_id is not None:
            if match_id not in self.match_ids:
                raise ValueError("Unknown match_id. Pass a game_key from player-match data.")
            current = self.history[self.history["game_key"] == match_id]
            target_date = pd.Timestamp(current["match_date"].iloc[0])
            round_name = current["round"].iloc[0]
            candidates = current[["player_id", "team", "opponent"]].drop_duplicates()
        else:
            if team is None or opponent is None or date is None:
                raise ValueError("Provide match_id, or provide team, opponent, and date.")
            target_date = _validated_date(date, self.minimum_date, self.maximum_date)
            team = _canonical_name(team, self.teams, "team")
            opponent = _canonical_name(opponent, self.teams, "opponent")
            if team == opponent:
                raise ValueError("team and opponent must be different teams.")
            round_name = self.history["round"].mode().iloc[0] if round_name is None else round_name
            candidates = self.history[
                (self.history["team"] == team)
                & (self.history["match_date"] < target_date)
            ][["player_id", "team"]].drop_duplicates("player_id")
            candidates["opponent"] = opponent
        if candidates.empty:
            raise ValueError("No eligible player history is available for this request.")
        rows, player_ids = [], []
        for candidate in candidates.itertuples(index=False):
            try:
                rows.append(self._features(candidate.player_id, candidate.team, candidate.opponent, target_date, round_name))
                player_ids.append(candidate.player_id)
            except ValueError:
                continue
        if not rows:
            raise ValueError("No eligible players have history before the requested date.")
        predictions = self.pipeline.predict(pd.DataFrame(rows))
        ranked = pd.DataFrame({"player_id": player_ids, "predicted_fantasy_points": predictions})
        ranked = ranked.sort_values(["predicted_fantasy_points", "player_id"], ascending=[False, True]).head(top_k)
        ranked["rank"] = range(1, len(ranked) + 1)
        return ranked[["rank", "player_id", "predicted_fantasy_points"]].to_dict("records")


match_winner_predictor = MatchWinnerPredictor(final_model, team_history, model_train)
top_player_predictor = TopPlayerPredictor(player_regressor, player_match_model)
joblib.dump(match_winner_predictor.pipeline, MATCH_PIPELINE_PATH)
joblib.dump(top_player_predictor.pipeline, PLAYER_PIPELINE_PATH)

print("PACKAGED MODEL INTERFACES")
print(f"Saved match pipeline: {MATCH_PIPELINE_PATH}")
print(f"Saved player pipeline: {PLAYER_PIPELINE_PATH}")
print("Match call: match_winner_predictor.predict_match_winner('Brisbane Lions', 'Melbourne Demons', '2025-05-18')")
print("Player call: top_player_predictor.predict_top_player(match_id='<game_key>', stat_type='fantasy_points', top_k=5)")

example_match = model_test.iloc[0]
print("\nVALID MATCH CALL:")
print(match_winner_predictor.predict_match_winner(
    str(example_match["home_team"]), str(example_match["away_team"]), example_match["match_date"],
    venue=example_match["venue"], round_name=example_match["round"]
))
example_game_id = str(player_match_model["game_key"].dropna().iloc[-1])
print("\nVALID TOP-PLAYER CALL:")
print(top_player_predictor.predict_top_player(match_id=example_game_id, top_k=3))
for invalid_call in [
    lambda: match_winner_predictor.predict_match_winner("Unknown Team", "Melbourne Demons", "2025-05-18"),
    lambda: match_winner_predictor.predict_match_winner("Brisbane Lions", "Melbourne Demons", "2035-05-18"),
    lambda: top_player_predictor.predict_top_player(match_id="missing-match-id"),
]:
    try:
        invalid_call()
    except ValueError as error:
        print(f"VALIDATION CHECK: {error}")

PACKAGED MODEL INTERFACES
Saved match pipeline: models\match_winner_pipeline.joblib
Saved player pipeline: models\top_player_pipeline.joblib
Match call: match_winner_predictor.predict_match_winner('Brisbane Lions', 'Melbourne Demons', '2025-05-18')
Player call: top_player_predictor.predict_top_player(match_id='<game_key>', stat_type='fantasy_points', top_k=5)

VALID MATCH CALL:
{'home_team': 'Adelaide Crows', 'away_team': 'Essendon Bombers', 'winner': 'Adelaide Crows', 'home_win_probability': 0.675123, 'away_win_probability': 0.324877, 'date': '2020-07-26'}

VALID TOP-PLAYER CALL:
[{'rank': 1, 'player_id': '45048', 'predicted_fantasy_points': 113.03501542794879}, {'rank': 2, 'player_id': '44673', 'predicted_fantasy_points': 111.34723371305864}, {'rank': 3, 'player_id': '43727', 'predicted_fantasy_points': 102.96257245571077}]
VALIDATION CHECK: Unknown team_a 'Unknown Team'. Check the training-data names.
VALIDATION CHECK: date must be between 1983-03-26 and 2025-09-27 because the fitte